In [52]:
import numpy as np 
import pandas as pd 

In [53]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_KEY = user_secrets.get_secret("wandb-key")


In [54]:
import wandb 
wandb.login(key=WB_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [55]:
import pandas as pd
import numpy as np
import re
import string

def load_data(train_path='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv', 
              test_path='/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'):
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    print(f"Train shape: {train_df.shape}")
    print(f"Test shape: {test_df.shape}")
    
    return train_df, test_df

train_df, test_df = load_data()


def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()

    text = text.translate(str.maketrans('', '', string.punctuation))

    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

Train shape: (2000, 8)
Test shape: (500, 7)


In [56]:
text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in text_columns:
    train_df[f'clean_{col}'] = train_df[col].apply(clean_text)
    test_df[f'clean_{col}'] = test_df[col].apply(clean_text)

print("Text cleaning complete. New columns generated.")

Text cleaning complete. New columns generated.


In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

all_train_text = train_df[[f'clean_{col}' for col in text_columns]].astype(str).agg(' '.join, axis=1)
tfidf.fit(all_train_text)

print(f"TF-IDF Vocabulary size: {len(tfidf.vocabulary_)}")

TF-IDF Vocabulary size: 2865


In [58]:
train_df[['prompt', 'A', 'B', 'C', 'D', 'E']].iloc[0].to_dict()

{'prompt': "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.",
 'A': "Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.",
 'B': 'Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.',
 'C': 'Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.',
 'D': 'Martin Heidegger be

In [59]:
def get_top_3_tfidf(row, vectorizer):
    prompt_text = row['clean_prompt']
    options_text = [row['clean_A'], row['clean_B'], row['clean_C'], row['clean_D'], row['clean_E']]
    labels = ['A', 'B', 'C', 'D', 'E']
    
    prompt_vec = vectorizer.transform([prompt_text])
    options_vec = vectorizer.transform(options_text)
    similarities = cosine_similarity(prompt_vec, options_vec).flatten()
    top_3_idx = np.argsort(similarities)[::-1][:3]
    top_3_labels = [labels[i] for i in top_3_idx]
    return " ".join(top_3_labels)
train_df['tfidf_prediction'] = train_df.apply(lambda row: get_top_3_tfidf(row, tfidf), axis=1)

print("Predictions generated. Example output:")
print(train_df[['id', 'tfidf_prediction']].head(3))

Predictions generated. Example output:
   id tfidf_prediction
0   1            C D B
1   2            C A B
2   3            E D C


In [60]:
def calculate_map_at_3(true_labels, predicted_labels_list):
    scores = []
    
    for true_label, preds in zip(true_labels, predicted_labels_list):
        pred_list = preds.split() 
        
        if true_label in pred_list:
            # Find the rank (1-indexed)
            rank = pred_list.index(true_label) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
            
    return np.mean(scores)

baseline_map3 = calculate_map_at_3(train_df['answer'], train_df['tfidf_prediction'])
print(f"Baseline TF-IDF MAP@3 Score: {baseline_map3:.4f}")

Baseline TF-IDF MAP@3 Score: 0.2387


In [61]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

train_df['tokens_prompt'] = train_df['clean_prompt'].apply(lambda x: x.split())
train_df['tokens_A'] = train_df['clean_A'].apply(lambda x: x.split())
train_df['tokens_B'] = train_df['clean_B'].apply(lambda x: x.split())
train_df['tokens_C'] = train_df['clean_C'].apply(lambda x: x.split())
train_df['tokens_D'] = train_df['clean_D'].apply(lambda x: x.split())
train_df['tokens_E'] = train_df['clean_E'].apply(lambda x: x.split())

all_tokens = pd.concat([
    train_df['tokens_prompt'], train_df['tokens_A'], 
    train_df['tokens_B'], train_df['tokens_C'], 
    train_df['tokens_D'], train_df['tokens_E']
]).tolist()

w2v_model = Word2Vec(sentences=all_tokens, vector_size=100, window=5, min_count=1, workers=4)

def get_sentence_embedding(tokens, model, vector_size):
    valid_words = [word for word in tokens if word in model.wv]
    if not valid_words:
        return np.zeros(vector_size)
    

    return np.mean([model.wv[word] for word in valid_words], axis=0)

def get_top_3_w2v(row, model):
    vector_size = model.vector_size
    prompt_vec = get_sentence_embedding(row['tokens_prompt'], model, vector_size).reshape(1, -1)
    
    options_tokens = [row['tokens_A'], row['tokens_B'], row['tokens_C'], row['tokens_D'], row['tokens_E']]
    labels = ['A', 'B', 'C', 'D', 'E']
    
    similarities = []
    for opt_tokens in options_tokens:
        opt_vec = get_sentence_embedding(opt_tokens, model, vector_size).reshape(1, -1)
        # Handle cases where vectors are all zeros
        if not np.any(prompt_vec) or not np.any(opt_vec):
            sim = 0.0
        else:
            sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        similarities.append(sim)
        
    top_3_idx = np.argsort(similarities)[::-1][:3]
    top_3_labels = [labels[i] for i in top_3_idx]
    
    return " ".join(top_3_labels)

train_df['w2v_prediction'] = train_df.apply(lambda row: get_top_3_w2v(row, w2v_model), axis=1)

w2v_map3 = calculate_map_at_3(train_df['answer'], train_df['w2v_prediction'])
print(f"Baseline Word2Vec MAP@3 Score: {w2v_map3:.4f}")

Baseline Word2Vec MAP@3 Score: 0.3248


In [62]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_transformer_embedding(text):
    if not isinstance(text, str) or text.strip() == "":
        return np.zeros(model.config.hidden_size)

    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    last_hidden_state = outputs.last_hidden_state
    
    attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * attention_mask
    summed = torch.sum(masked_embeddings, dim=1)
    counts = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
    
    sentence_embedding = summed / counts
    return sentence_embedding.numpy().flatten()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [63]:
import torch
import numpy as np
from tqdm.notebook import tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

if torch.cuda.device_count() > 1:
    print(f"Utilizing {torch.cuda.device_count()} GPUs for parallel processing!")
    model = torch.nn.DataParallel(model)


def get_batched_embeddings(texts, batch_size=128, show_progress=False):
    texts = [str(t) if pd.notna(t) else "" for t in texts]
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding Batches", disable=not show_progress):
        batch = texts[i:i + batch_size]
        
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        last_hidden_state = outputs.last_hidden_state
        attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden_state.size()).float()
        
        masked_embeddings = last_hidden_state * attention_mask
        summed = torch.sum(masked_embeddings, dim=1)
        counts = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
        
        sentence_embeddings = summed / counts
        
        all_embeddings.append(sentence_embeddings.cpu().numpy())
        
    return np.vstack(all_embeddings)

Utilizing 2 GPUs for parallel processing!


In [64]:
emb_prompts = get_batched_embeddings(train_df['clean_prompt'].tolist(), show_progress=True)

option_embeddings = {}
for opt in ['A', 'B', 'C', 'D', 'E']:
    print(f"Embedding Option {opt}...")
    option_embeddings[opt] = get_batched_embeddings(train_df[f'clean_{opt}'].tolist(), show_progress=True)

similarities = []
norm_prompts = np.linalg.norm(emb_prompts, axis=1, keepdims=True)

for opt in ['A', 'B', 'C', 'D', 'E']:
    emb_opt = option_embeddings[opt]
    norm_opt = np.linalg.norm(emb_opt, axis=1, keepdims=True)
    
    sim = np.sum(emb_prompts * emb_opt, axis=1, keepdims=True) / np.maximum(norm_prompts * norm_opt, 1e-9)
    similarities.append(sim)

similarities_matrix = np.concatenate(similarities, axis=1)


top_3_idx = np.argsort(similarities_matrix, axis=1)[:, ::-1][:, :3]
labels_array = np.array(['A', 'B', 'C', 'D', 'E'])

train_df['transformer_prediction'] = [" ".join(labels) for labels in labels_array[top_3_idx]]


transformer_map3 = calculate_map_at_3(train_df['answer'], train_df['transformer_prediction'])
print(f"Transformer MAP@3: {transformer_map3:.4f}")

Embedding Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding Option A...


Embedding Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding Option B...


Embedding Batches:   0%|          | 0/16 [00:00<?, ?it/s]

...

Embedding Option D...


Embedding Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding Option E...


Embedding Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Transformer MAP@3: 0.4081


In [65]:
from transformers import pipeline

zero_shot_classifier = pipeline(
    "zero-shot-classification", 
    model="valhalla/distilbart-mnli-12-3", 
    device=0
)

def predict_zero_shot_sample(row):
    prompt_text = row['prompt']
    options = [row['A'], row['B'], row['C'], row['D'], row['E']]
    labels = ['A', 'B', 'C', 'D', 'E']
    
    option_to_label = {opt: label for opt, label in zip(options, labels)}
    
    result = zero_shot_classifier(prompt_text, candidate_labels=options)
    
    top_3_options = result['labels'][:3]
    top_3_labels = [option_to_label[opt] for opt in top_3_options]
    
    return " ".join(top_3_labels)

sample_preds = train_df.head(5).apply(predict_zero_shot_sample, axis=1)

for idx, (pred, actual) in enumerate(zip(sample_preds, train_df['answer'].head(5))):
    print(f"Row {idx+1} | Predicted Top 3: {pred} | Actual Answer: {actual}")

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Row 1 | Predicted Top 3: C D E | Actual Answer: B
Row 2 | Predicted Top 3: D A E | Actual Answer: A
Row 3 | Predicted Top 3: B A C | Actual Answer: C
Row 4 | Predicted Top 3: C D E | Actual Answer: B
Row 5 | Predicted Top 3: B A C | Actual Answer: A


# Milestone 3 

In [66]:
!pip install -q langchain-text-splitters langchain-huggingface faiss-cpu langchain-community

In [67]:
import glob
import faiss
from tqdm.notebook import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS_DIR = "/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus"
txt_files = glob.glob(os.path.join(CORPUS_DIR, "*.txt"))
print(f"Found {len(txt_files)} text files.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=150)
all_chunks = []

for file_path in tqdm(txt_files, desc="Chunking Files"):
    filename = os.path.basename(file_path).replace('.txt', '')
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
        
    if text.strip():
        chunks = text_splitter.split_text(text)

        for chunk in chunks:
            all_chunks.append(f"Source: {filename}\n{chunk}")

print(f"Generated {len(all_chunks)} chunks for embedding.")


print("Embedding external knowledge base... (This may take a minute or two)")

corpus_embeddings = get_batched_embeddings(all_chunks, batch_size=256, show_progress=True) 


dimension = corpus_embeddings.shape[1]  
index = faiss.IndexFlatIP(dimension)    
faiss.normalize_L2(corpus_embeddings)   
index.add(corpus_embeddings)
print(f"FAISS Index loaded with {index.ntotal} vectors.")


def retrieve_real_context(query_text, top_k=3):
    query_vec = get_batched_embeddings([query_text], batch_size=1, show_progress=False)
    faiss.normalize_L2(query_vec)
    
    distances, indices = index.search(query_vec, top_k)
    
    retrieved_texts = [all_chunks[idx] for idx in indices[0]]
    return "\n\n".join(retrieved_texts)

def format_rag_mcq(row):
    prompt = row['prompt']
    context = retrieve_real_context(prompt, top_k=3)
    formatted_text = f"Context:\n{context}\n\nQuestion: {prompt}\n\nOptions:\nA: {row['A']}\nB: {row['B']}\nC: {row['C']}\nD: {row['D']}\nE: {row['E']}"
    return formatted_text


sample_prompt = train_df.loc[0, 'prompt']
print(f"PROMPT: {sample_prompt}")
print(f"\nRETRIEVED CONTEXT:\n{retrieve_real_context(sample_prompt)}")

Found 533 text files.


Chunking Files:   0%|          | 0/533 [00:00<?, ?it/s]

Generated 70529 chunks for embedding.
Embedding external knowledge base... (This may take a minute or two)


Embedding Batches:   0%|          | 0/276 [00:00<?, ?it/s]

FAISS Index loaded with 70529 vectors.
PROMPT: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


Embedding Batches:   0%|          | 0/1 [00:00<?, ?it/s]


RETRIEVED CONTEXT:
Source: Heideggerian_terminology
it is revealed as a threefold condition of Being. Time, the present, and the notion of the "eternal", are modes of temporality, which is the way humanity views time. For Heidegger, it is very different from the mistaken view of time as being a linear series of past, present and future. Instead he sees it as being an ecstasy, an outside-of-itself, of futural projections (possibilities) and one's place in history as a part of one's generation. Possibilities, then, are integral to understanding of

Source: F__C__S__Schiller
the reference to Time could not, of course, be recovered, any more than the individuality of Reality can be deduced, when once ignored. The assumption is made that, to express the 'truth' about Reality, its 'thisness,' individuality, change and its immersion in a certain temporal and spatial environment may be neglected, and the timeless validity of a conception is thus substituted for the living, changing and perish

In [68]:
sample_rag_input = format_rag_mcq(train_df.iloc[0])
print("\n--- FORMATTED RAG INPUT ---")
print(sample_rag_input)

Embedding Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- FORMATTED RAG INPUT ---
Context:
Source: Heideggerian_terminology
it is revealed as a threefold condition of Being. Time, the present, and the notion of the "eternal", are modes of temporality, which is the way humanity views time. For Heidegger, it is very different from the mistaken view of time as being a linear series of past, present and future. Instead he sees it as being an ecstasy, an outside-of-itself, of futural projections (possibilities) and one's place in history as a part of one's generation. Possibilities, then, are integral to understanding of

Source: F__C__S__Schiller
the reference to Time could not, of course, be recovered, any more than the individuality of Reality can be deduced, when once ignored. The assumption is made that, to express the 'truth' about Reality, its 'thisness,' individuality, change and its immersion in a certain temporal and spatial environment may be neglected, and the timeless validity of a conception is thus substituted for the living, c

# Milestone 4 

In [69]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset as TorchDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset as HFDataset

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
LABELS = OPTION_COLS
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

train_split_df, val_split_df = train_test_split(
    train_df, test_size=0.1, random_state=42, stratify=train_df['answer']
)
train_split_df = train_split_df.reset_index(drop=True)
val_split_df = val_split_df.reset_index(drop=True)

y_train_split = np.array([LABEL2ID[a] for a in train_split_df['answer']])
y_val_split = np.array([LABEL2ID[a] for a in val_split_df['answer']])

def compute_metrics_mc(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    top3_idx = np.argsort(-predictions, axis=1)[:, :3]

    map3_scores = []
    for i, true_l in enumerate(labels):
        if true_l in top3_idx[i]:
            rank = list(top3_idx[i]).index(true_l) + 1
            map3_scores.append(1.0 / rank)
        else:
            map3_scores.append(0.0)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
        "map3": float(np.mean(map3_scores)),
    }

def row_text(df):
    return (df['clean_prompt'] + " " + df['clean_A'] + " " + df['clean_B'] + " " +
            df['clean_C'] + " " + df['clean_D'] + " " + df['clean_E']).tolist()

X_train_tfidf = tfidf.transform(row_text(train_split_df)).toarray()
X_val_tfidf = tfidf.transform(row_text(val_split_df)).toarray()
X_test_tfidf = tfidf.transform(row_text(test_df)).toarray()

class MCQScratchDataset(TorchDataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(MCQScratchDataset(X_train_tfidf, y_train_split), batch_size=16, shuffle=True)
val_loader = DataLoader(MCQScratchDataset(X_val_tfidf, y_val_split), batch_size=16, shuffle=False)

class ScratchMCQSolver(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 5)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model_scratch = ScratchMCQSolver(input_dim=X_train_tfidf.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=1e-3)

wandb.init(project="22f2000691-t22026", name="Model-1-Scratch-MLP")
for epoch in range(10):
    model_scratch.train()
    total_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model_scratch(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == by).sum().item()
        total += by.size(0)

    model_scratch.eval()
    val_preds, val_labels, val_logits_all = [], [], []
    with torch.no_grad():
        for bx, by in val_loader:
            out = model_scratch(bx.to(device))
            val_preds.extend(out.argmax(1).cpu().tolist())
            val_labels.extend(by.tolist())
            val_logits_all.append(out.cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average="macro")

    val_logits_all = np.concatenate(val_logits_all, axis=0)
    top3_idx = np.argsort(-val_logits_all, axis=1)[:, :3]
    map3_scores = [1.0/(list(top3_idx[i]).index(l)+1) if l in top3_idx[i] else 0.0
                   for i, l in enumerate(val_labels)]
    val_map3 = float(np.mean(map3_scores))

    wandb.log({"epoch": epoch+1, "train_loss": total_loss/len(train_loader),
               "train_accuracy": correct/total, "eval_accuracy": val_acc,
               "eval_f1": val_f1, "eval_map3": val_map3})
    print(f"Epoch {epoch+1}: val_acc={val_acc:.4f} val_f1={val_f1:.4f} val_map3={val_map3:.4f}")

model1_metrics = {"accuracy": val_acc, "f1": val_f1, "map3": val_map3}
wandb.finish()

Epoch 1: val_acc=1.0000 val_f1=1.0000 val_map3=1.0000
Epoch 2: val_acc=1.0000 val_f1=1.0000 val_map3=1.0000
Epoch 3: val_acc=1.0000 val_f1=1.0000 val_map3=1.0000
Epoch 4: val_acc=1.0000 val_f1=1.0000 val_map3=1.0000
Epoch 5: val_acc=1.0000 val_f1=1.0000 val_map3=1.0000
Epoch 6: val_acc=1.0000 val_f1=1.0000 val_map3=1.0000
Epoch 7: val_acc=1.0000 val_f1=1.0000 val_map3=1.0000
Epoch 8: val_acc=1.0000 val_f1=1.0000 val_map3=1.0000
Epoch 9: val_acc=1.0000 val_f1=1.0000 val_map3=1.0000
Epoch 10: val_acc=1.0000 val_f1=1.0000 val_map3=1.0000


epoch,▁▂▃▃▄▅▆▆▇█
eval_accuracy,▁▁▁▁▁▁▁▁▁▁
eval_f1,▁▁▁▁▁▁▁▁▁▁
eval_map3,▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁█████████
train_loss,█▂▁▁▁▁▁▁▁▁
epoch,10
eval_accuracy,1
eval_f1,1
eval_map3,1
train_accuracy,1


In [70]:
train_split_df['context'] = train_split_df['prompt'].apply(lambda p: retrieve_real_context(p, top_k=2))
val_split_df['context'] = val_split_df['prompt'].apply(lambda p: retrieve_real_context(p, top_k=2))
test_df['context'] = test_df['prompt'].apply(lambda p: retrieve_real_context(p, top_k=2))

tokenizer2 = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_mcq_with_rag(examples):
    first_sentences = [[f"Context: {ctx}\nQuestion: {p}"] * 5
                        for ctx, p in zip(examples['context'], examples['prompt'])]
    second_sentences = [[f"A: {a}", f"B: {b}", f"C: {c}", f"D: {d}", f"E: {e}"]
                         for a,b,c,d,e in zip(examples['A'],examples['B'],examples['C'],examples['D'],examples['E'])]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    tokenized = tokenizer2(first_sentences, second_sentences, truncation=True, max_length=384)
    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}

hf_train_ds = HFDataset.from_pandas(train_split_df)
encoded_train_ds = hf_train_ds.map(preprocess_mcq_with_rag, batched=True, remove_columns=hf_train_ds.column_names)
encoded_train_ds = encoded_train_ds.add_column("label", y_train_split.tolist())

hf_val_ds = HFDataset.from_pandas(val_split_df)
encoded_val_ds = hf_val_ds.map(preprocess_mcq_with_rag, batched=True, remove_columns=hf_val_ds.column_names)
encoded_val_ds = encoded_val_ds.add_column("label", y_val_split.tolist())

hf_test_ds = HFDataset.from_pandas(test_df)
encoded_test_ds = hf_test_ds.map(preprocess_mcq_with_rag, batched=True, remove_columns=hf_test_ds.column_names)

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: any
    padding: Union[bool, str] = True
    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [f.pop(label_name) for f in features] if label_name in features[0] else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flat = [[{k: v[i] for k, v in f.items()} for i in range(num_choices)] for f in features]
        flat = sum(flat, [])
        batch = self.tokenizer.pad(flat, padding=self.padding, return_tensors="pt")
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer2)

model2 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME, use_safetensors=True)

training_args_m2 = TrainingArguments(
    output_dir="./deberta_mcq_results",
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=1e-5, warmup_steps=50,
    per_device_train_batch_size=2, per_device_eval_batch_size=4,
    gradient_accumulation_steps=8, num_train_epochs=3,
    weight_decay=0.01, adam_epsilon=1e-6,
    logging_steps=10, max_grad_norm=0.5, fp16=False,
    report_to="wandb", run_name="Model-2-DeBERTa-v3-Full",
    load_best_model_at_end=True, metric_for_best_model="map3",
)

trainer_m2 = Trainer(model=model2, args=training_args_m2,
                      train_dataset=encoded_train_ds, eval_dataset=encoded_val_ds,
                      processing_class=tokenizer2, data_collator=data_collator,
                      compute_metrics=compute_metrics_mc)
trainer_m2.train()
model2_metrics = trainer_m2.evaluate()
wandb.finish()

Embedding Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

eval/accuracy,▆█▂▁
eval/f1,█▆▄▁
eval/loss,▁▁▁▁
eval/map3,▇█▁█
eval/runtime,█▃▁▆
eval/samples_per_second,▁▆█▃
eval/steps_per_second,▁▆█▃
train/epoch,▁▁▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇█████
train/global_step,▁▁▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇█████
train/grad_norm,▄▂▁█▁▁▁▁▁▁▁▁▁▁▁▁▁
+2,...


In [71]:
model3_base = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME, use_safetensors=True)

lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["classifier", "pooler"],  # <-- the fix: keeps the classification head trainable
)
model3 = get_peft_model(model3_base, lora_config)
model3.print_trainable_parameters()

training_args_m3 = TrainingArguments(
    output_dir="./deberta_lora_results",
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=5e-5, warmup_steps=50,
    per_device_train_batch_size=2, per_device_eval_batch_size=4,
    gradient_accumulation_steps=8, num_train_epochs=3,
    logging_steps=10, max_grad_norm=0.5, fp16=False,
    report_to="wandb", run_name="Model-3-DeBERTa-LoRA",
    load_best_model_at_end=True, metric_for_best_model="map3",
)

trainer_m3 = Trainer(model=model3, args=training_args_m3,
                      train_dataset=encoded_train_ds, eval_dataset=encoded_val_ds,
                      processing_class=tokenizer2, data_collator=data_collator,
                      compute_metrics=compute_metrics_mc)
trainer_m3.train()
model3_metrics = trainer_m3.evaluate()
wandb.finish()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                

trainable params: 886,273 || all params: 185,309,186 || trainable%: 0.4783


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1,Map3
1,26.437305,3.095703,0.320000,0.303696,0.492500
2,55.553711,3.242188,0.210000,0.203049,0.342500
3,0.000000,nan,0.185000,0.062447,0.384167


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


eval/accuracy,█▂▁█
eval/f1,█▅▁█
eval/loss,▁█ ▁
eval/map3,█▁▃█
eval/runtime,██▁█
eval/samples_per_second,▁▁█▁
eval/steps_per_second,▁▁█▁
train/epoch,▁▁▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇█████
train/global_step,▁▁▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇█████
train/grad_norm,▁▁▃▂▂█▇█▅▃▆█▄
+2,...


# Milestone 5

In [72]:
import torch.nn.functional as F

wandb.init(project="22f2000691-t22026", name="Model-Final-Ensemble-Inference")

model_scratch.eval()
with torch.no_grad():
    logits_m1_val = model_scratch(torch.tensor(X_val_tfidf, dtype=torch.float32).to(device)).cpu().numpy()
    logits_m1_test = model_scratch(torch.tensor(X_test_tfidf, dtype=torch.float32).to(device)).cpu().numpy()

logits_m2_val = trainer_m2.predict(encoded_val_ds).predictions
logits_m2_test = trainer_m2.predict(encoded_test_ds).predictions
logits_m3_val = trainer_m3.predict(encoded_val_ds).predictions
logits_m3_test = trainer_m3.predict(encoded_test_ds).predictions

map3_scores = np.array([model1_metrics["map3"], model2_metrics["eval_map3"], model3_metrics["eval_map3"]])
weights = map3_scores / map3_scores.sum() if map3_scores.sum() > 1e-8 else np.array([1/3, 1/3, 1/3])
w1, w2, w3 = weights
print(f"Ensemble weights (MAP@3-based) -> M1: {w1:.3f}, M2: {w2:.3f}, M3: {w3:.3f}")

T = 1.2
probs_m1_val = F.softmax(torch.tensor(logits_m1_val)/T, dim=-1).numpy()
probs_m2_val = F.softmax(torch.tensor(logits_m2_val)/T, dim=-1).numpy()
probs_m3_val = F.softmax(torch.tensor(logits_m3_val)/T, dim=-1).numpy()
ensemble_probs_val = w1*probs_m1_val + w2*probs_m2_val + w3*probs_m3_val


def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    return 1/(predicted.index(actual)+1) if actual in predicted else 0

top3_val = [[LABELS[i] for i in np.argsort(-row)[:3]] for row in ensemble_probs_val]
actual_val = [LABELS[i] for i in y_val_split]
val_map3 = np.mean([apk(a, p) for a, p in zip(actual_val, top3_val)])
print(f"Ensemble Validation MAP@3: {val_map3:.4f}")
wandb.log({"ensemble_val_map3": val_map3, "weight_m1": w1, "weight_m2": w2, "weight_m3": w3})

probs_m1_test = F.softmax(torch.tensor(logits_m1_test)/T, dim=-1).numpy()
probs_m2_test = F.softmax(torch.tensor(logits_m2_test)/T, dim=-1).numpy()
probs_m3_test = F.softmax(torch.tensor(logits_m3_test)/T, dim=-1).numpy()
ensemble_probs_test = w1*probs_m1_test + w2*probs_m2_test + w3*probs_m3_test

final_predictions = [" ".join(LABELS[i] for i in np.argsort(-row)[:3]) for row in ensemble_probs_test]
id_col = 'ID' if 'ID' in test_df.columns else 'id'
submission_df = pd.DataFrame({'ID': test_df[id_col], 'Prediction': final_predictions})
submission_df.to_csv('submission.csv', index=False)
wandb.finish()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Ensemble weights (MAP@3-based) -> M1: 0.531, M2: 0.207, M3: 0.262
Ensemble Validation MAP@3: 1.0000


ensemble_val_map3,▁
test/accuracy,▁█
test/f1,▁█
test/loss,█▁
test/map3,▁█
test/runtime,▁▆▂█
test/samples_per_second,██▁▁
test/steps_per_second,██▁▂
weight_m1,▁
weight_m2,▁
+1,...
